# BPLLM trên Kaggle: chạy LLM và chấm điểm (oracle)

Notebook gộp: chọn model **int4** hoặc **int8**, bộ câu hỏi và kiểu prompt ở **Khối 1**, các khối sau không cần sửa.

**Trước khi chạy, cài đặt ở cột bên phải của notebook:**
1. **Accelerator:** GPU T4 x2
2. **Internet:** On
3. **Input:** Add Input, chọn dataset chứa 2 file top-10 của bộ câu hỏi muốn chạy

**Đổi cấu hình giữa chừng** (ví dụ int4 sang int8): sửa Khối 1 rồi chạy lại từ Khối 1 xuống hết. File kết quả mang tên cấu hình nên các lần chạy không ghi đè nhau.

## Khối 1. Cấu hình

| Biến | Giá trị | Nghĩa |
|---|---|---|
| `QUANT` | `"int4"`, `"int8"` | Model của tác giả, fine-tune với PEFT int4 hoặc int8 |
| `QUESTION_SET` | `"ft"`, `"refined"` | Bộ câu hỏi 6 (`food_delivery_ft_top10*.jsonl`) hoặc bộ 3 (`food_delivery_top10*.jsonl`) |
| `PROMPT_STYLE` | `"chat"`, `"raw"` | `chat`: qua chat template, như model card. `raw`: văn bản thô, kết thúc bằng `Helpful Answer:` |

In [ ]:
QUANT = "int4"          # "int4" hoặc "int8"
QUESTION_SET = "ft"     # "ft" (bộ 6) hoặc "refined" (bộ 3)
PROMPT_STYLE = "chat"   # "chat" hoặc "raw"
MAX_NEW_TOKENS = 128

MODELS = {
    "int4": "angeloc1/llama3dot1FoodDel4v05",
    "int8": "angeloc1/llama3dot1FoodDel8v02",
}
QUESTION_FILES = {
    "ft": {"specific": "food_delivery_ft_top10.jsonl",
           "enriched": "food_delivery_ft_top10_enriched.jsonl"},
    "refined": {"specific": "food_delivery_top10.jsonl",
                "enriched": "food_delivery_top10_enriched.jsonl"},
}

assert QUANT in MODELS, f"QUANT phải là một trong {list(MODELS)}"
assert QUESTION_SET in QUESTION_FILES, f"QUESTION_SET phải là một trong {list(QUESTION_FILES)}"
assert PROMPT_STYLE in ("chat", "raw"), "PROMPT_STYLE phải là 'chat' hoặc 'raw'"

MODEL_PATH = MODELS[QUANT]
RUN_TAG = f"{QUANT}_{QUESTION_SET}_{PROMPT_STYLE}"   # gắn vào tên file kết quả
print("Model      :", MODEL_PATH)
print("Bộ câu hỏi :", QUESTION_SET, QUESTION_FILES[QUESTION_SET])
print("Kiểu prompt:", PROMPT_STYLE)
print("Tên lần chạy:", RUN_TAG)

## Khối 2. Kiểm tra GPU

Phải thấy bảng có Tesla T4. Nếu báo lỗi thì chưa bật Accelerator.

In [ ]:
!nvidia-smi

## Khối 3. Cài thư viện

In [ ]:
!pip install -q -U transformers accelerate

## Khối 4. Tìm và đọc 2 file top-10

`specific` là chunk theo bài báo, `enriched` là chunk cải tiến. Mỗi file phải có 104 câu hỏi.

In [ ]:
import glob
import json

INPUT_DIR = "/kaggle/input"
OUTPUT_DIR = "/kaggle/working"


def find_input(name):
  matches = glob.glob(f"{INPUT_DIR}/**/{name}", recursive=True)
  assert matches, f"Không thấy {name} trong {INPUT_DIR}. Hãy Add Input dataset chứa file này."
  return matches[0]


def read_jsonl(path):
  with open(path, encoding="utf-8") as f:
    return [json.loads(line) for line in f if line.strip()]


INPUT_FILES = {name: find_input(filename)
               for name, filename in QUESTION_FILES[QUESTION_SET].items()}
for name, path in INPUT_FILES.items():
  print(f"{name}: {len(read_jsonl(path))} câu hỏi, {path}")

## Khối 5. Nạp model

Nạp ở `float16`, đúng như model card (trọng số của cả hai model đều lưu dạng F16). Lần đầu tải khoảng 16 GB nên mất vài phút.

Nếu đang có model khác trong bộ nhớ (vừa đổi `QUANT`), khối này giải phóng nó trước khi nạp model mới.

In [ ]:
import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if "model" in globals():
  del model
  gc.collect()
  torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    device_map="auto",
    torch_dtype=torch.float16,
).eval()
LOADED_MODEL = MODEL_PATH
print("Đã nạp:", LOADED_MODEL)

## Khối 6. Dựng prompt và hàm hỏi model

Phần nội dung prompt giống nhau ở cả hai kiểu (Hình 4 của bài báo: câu lệnh, 10 chunk, câu hỏi). Khác nhau ở cách gửi:

- `chat`: prompt là một tin nhắn của `user`, đi qua chat template của Llama 3.1.
- `raw`: prompt gửi nguyên dạng văn bản, thêm dòng `Helpful Answer:` ở cuối; phần model tự viết tiếp sau `Question:` bị cắt bỏ.

Mỗi câu hỏi là một lần hỏi độc lập; model không nhận `expected`.

In [ ]:
PROMPT = ("Use the following pieces of context to answer the question at the end.\n"
          "Answer 'yes' if true or 'no' if false.\n\n"
          "{context}\n\n"
          "Question: {question}")


def build_prompt(record):
  context = "\n\n".join(chunk["text"] for chunk in record["contexts"])
  prompt = PROMPT.format(context=context, question=record["question"])
  if PROMPT_STYLE == "raw":
    prompt += "\nHelpful Answer:"
  return prompt


def ask(record):
  assert LOADED_MODEL == MODEL_PATH, "Đã đổi QUANT: chạy lại Khối 5 để nạp đúng model."
  prompt = build_prompt(record)
  if PROMPT_STYLE == "chat":
    inputs = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=True,
        add_generation_prompt=True, return_dict=True, return_tensors="pt")
  else:
    inputs = tokenizer(prompt, return_tensors="pt")
  inputs = inputs.to(model.device)
  with torch.no_grad():
    output_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                                pad_token_id=tokenizer.eos_token_id)
  text = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[-1]:],
                          skip_special_tokens=True)
  if PROMPT_STYLE == "raw":
    text = text.split("\nQuestion:")[0]   # bỏ phần model tự viết tiếp
  return text.strip()

## Khối 7. Thử 1 câu hỏi

Xem prompt và câu trả lời có hợp lý không trước khi chạy cả 104 câu.

In [ ]:
sample = read_jsonl(INPUT_FILES["specific"])[0]
print(build_prompt(sample))
print("-" * 70)
print("Model trả lời:", ask(sample))
print("Đáp án đúng  :", sample["expected"])

## Khối 8. Sinh câu trả lời cho 104 câu, mỗi loại chunk một file

Kết quả ghi vào `/kaggle/working/answers_<tên lần chạy>_specific.jsonl` và `..._enriched.jsonl`.

In [ ]:
from tqdm.auto import tqdm


def run(name):
  output_file = f"{OUTPUT_DIR}/answers_{RUN_TAG}_{name}.jsonl"
  with open(output_file, "w", encoding="utf-8") as f:
    for record in tqdm(read_jsonl(INPUT_FILES[name]), desc=f"{RUN_TAG} {name}"):
      row = {"id": record["id"], "question": record["question"],
             "expected": record["expected"], "answer": ask(record)}
      f.write(json.dumps(row, ensure_ascii=False) + "\n")
  return output_file


ANSWER_FILES = {name: run(name) for name in INPUT_FILES}

## Khối 9. Oracle

Giống `src/bpllm/evaluation/oracle.py`: lấy `yes` hoặc `no` xuất hiện đầu tiên trong câu trả lời rồi so với `expected`. Câu không rút được yes/no tính là sai và được liệt kê riêng.

In [ ]:
import re

_YES_NO = re.compile(r"\b(yes|no)\b", re.IGNORECASE)


def extract_answer(text):
  match = _YES_NO.search(text or "")
  return match.group(1).lower() if match else None


def grade(records):
  total = {"yes": 0, "no": 0}
  correct = {"yes": 0, "no": 0}
  wrong, unparsed = [], []
  for i, record in enumerate(records, 1):
    qid = record.get("id", i)
    expected = str(record["expected"]).strip().lower()
    predicted = extract_answer(record.get("answer", ""))
    total[expected] += 1
    if predicted is None:
      unparsed.append(qid)
    if predicted == expected:
      correct[expected] += 1
    else:
      wrong.append(qid)
  n = len(records)
  return {
      "total": n,
      "correct": sum(correct.values()),
      "accuracy": sum(correct.values()) / n if n else 0.0,
      "accuracy_yes": correct["yes"] / total["yes"] if total["yes"] else 0.0,
      "accuracy_no": correct["no"] / total["no"] if total["no"] else 0.0,
      "wrong_ids": wrong,
      "unparsed_ids": unparsed,
  }


def evaluate(name):
  records = read_jsonl(ANSWER_FILES[name])
  report = grade(records)
  print(f"[{RUN_TAG} / {name}] accuracy {report['accuracy']:.2%} "
        f"({report['correct']}/{report['total']}), "
        f"yes {report['accuracy_yes']:.2%}, no {report['accuracy_no']:.2%}")
  print(f"  Không rút được yes/no: {report['unparsed_ids']}")
  for record in records:
    if record["id"] in report["wrong_ids"]:
      print(f"  Sai câu {record['id']} (đúng là {record['expected']}): {record['question']}")
      print(f"      Model: {record['answer'][:200]}")
  return report


REPORTS = {}

## Khối 10. Oracle cho chunk theo bài báo (specific)

In [ ]:
REPORTS["specific"] = evaluate("specific")

## Khối 11. Oracle cho chunk enriched

In [ ]:
REPORTS["enriched"] = evaluate("enriched")

## Khối 12. So sánh và lưu báo cáo

File kết quả nằm ở mục **Output** (`/kaggle/working`) ở cột bên phải. Báo cáo ghi kèm cấu hình của lần chạy.

In [ ]:
print(f"Lần chạy: {RUN_TAG} ({MODEL_PATH})")
print(f"{'chunking':<10} {'accuracy':>9} {'đúng':>8} {'yes':>8} {'no':>8}")
for name, report in REPORTS.items():
  print(f"{name:<10} {report['accuracy']:>9.2%} {report['correct']:>4}/{report['total']:<3} "
        f"{report['accuracy_yes']:>8.2%} {report['accuracy_no']:>8.2%}")

summary = {
    "config": {"quant": QUANT, "model": MODEL_PATH, "question_set": QUESTION_SET,
               "prompt_style": PROMPT_STYLE, "max_new_tokens": MAX_NEW_TOKENS,
               "input_files": INPUT_FILES},
    "reports": REPORTS,
}
report_file = f"{OUTPUT_DIR}/accuracy_report_{RUN_TAG}.json"
with open(report_file, "w", encoding="utf-8") as f:
  json.dump(summary, f, indent=2, ensure_ascii=False)
print("Đã lưu:", report_file)